# Stage 5 - adversarial training under two threat models (the decisive test)

Earlier ad-hoc analysis evaluated the defence *differently* on each dataset (transfer on one, white-box on the other), which is not a fair comparison. Here both datasets are evaluated the SAME way, under BOTH threat models, so 'does adversarial training help or hurt, and does it depend on the dataset' can be answered honestly.

- **transfer**: attack crafted from the *baseline* CNN and fed to every model (a real attacker has no access to the defended model).
- **white-box**: attack crafted against the model it hits (worst case; the Random Forest has no gradients, so white-box does not apply to it).

In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
DATASETS = ['ciciov2024', 'road']

## Train baseline + defended CNN + RF, then attack both ways at PGD eps=0.10

In [ ]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
from adversec.experiments.defense import adversarial_train_cnn
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p, labels=None): return f1_score(y, p, labels=labels, average='macro', zero_division=0)

results = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    # robust-support = classes with >=2 test frames (computed, not hard-coded -> equal for both datasets)
    counts = pd.Series(yte).value_counts(); rs = [i for i in range(len(classes)) if counts.get(i, 0) >= 2]

    print('########## ' + name + ' ##########')
    base = train_cnn(CNN1D(Xtr.shape[1], len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    print('-- adversarial training (PGD-augmented) --')
    defended = adversarial_train_cnn(Xtr, ytr, strategy='pgd', n_epochs=50, device=DEVICE, class_weights=cw)
    rf = build_random_forest().fit(Xtr, ytr)
    base_clf = atk.wrap_cnn_for_art(base, Xtr.shape[1], len(classes), DEVICE)
    def_clf  = atk.wrap_cnn_for_art(defended, Xtr.shape[1], len(classes), DEVICE)

    # two threat models at PGD eps=0.10
    X_transfer = atk.generate_pgd(base_clf, Xte, 0.10)   # crafted from the BASELINE (transfer)
    X_wb_def   = atk.generate_pgd(def_clf,  Xte, 0.10)   # crafted from the DEFENDED model (white-box)

    def sc(clf, X): p = clf.predict(X).argmax(1); return (mf1(yte, p), mf1(yte, p, rs))
    def sc_rf(X):   p = rf.predict(X);            return (mf1(yte, p), mf1(yte, p, rs))
    def fmt(s): return 'n/a' if s[0] is None else f'{s[0]:.3f} | {s[1]:.3f}'

    rows = [
        ('baseline CNN', sc(base_clf, Xte), sc(base_clf, X_transfer), sc(base_clf, X_transfer)),  # wb == transfer for baseline
        ('defended CNN', sc(def_clf,  Xte), sc(def_clf,  X_transfer), sc(def_clf,  X_wb_def)),
        ('Random Forest', sc_rf(Xte),        sc_rf(X_transfer),        (None, None)),
    ]
    print(name + ': PGD eps=0.10   (macro-F1 | robust-support-F1)')
    print(f"  {'model':15s}{'clean':>16}{'transfer':>16}{'white-box':>16}")
    for lab, cl, tr, wb in rows:
        print(f"  {lab:15s}{fmt(cl):>16}{fmt(tr):>16}{fmt(wb):>16}")
    results[name] = {'clean': sc(def_clf, Xte), 'transfer': sc(def_clf, X_transfer), 'whitebox': sc(def_clf, X_wb_def)}

## The decisive comparison

In [ ]:
print('DEFENDED CNN - the decisive 2x2 (macro-F1 under PGD eps=0.10)')
print(f"  {'dataset':12s}{'clean':>10}{'transfer':>10}{'white-box':>10}")
for name in DATASETS:
    d = results[name]
    print(f"  {name:12s}{d['clean'][0]:>10.3f}{d['transfer'][0]:>10.3f}{d['whitebox'][0]:>10.3f}")
print('Read: same result on both datasets -> no contingency. Diverges -> real, and now consistently measured.')